# Sparkle V2 — Fase 2+3: Extracción de landmarks faciales (MediaPipe Tasks API)

Este notebook asume que ya corriste la Fase 0+1 (dataset descargado en `/content/daisee_raw`, labels copiados a Drive).

**Nota de versión:** MediaPipe deprecó la API "legacy" (`mp.solutions.face_mesh`) en 2023; las versiones actuales del paquete ya no la incluyen. Este notebook usa la **Tasks API** (`FaceLandmarker`), que es la vigente. Como beneficio extra, esta API calcula directamente la matriz de transformación facial, de la que sacamos yaw/pitch/roll sin necesitar `solvePnP` manual.

**Qué hace:**
1. Para cada clip de DAiSEE, muestrea 60 frames distribuidos uniformemente (10s a 30fps → coincide con el benchmark de Gothwal et al. 2025 que citás en la tesis).
2. Corre MediaPipe FaceLandmarker sobre cada frame **sin persistir la imagen** (se procesa y se descarta).
3. Calcula por frame: apertura ocular (EAR) y orientación de cabeza (yaw, pitch, roll).
4. Guarda por clip una secuencia `(60, 5)` en un `.npz` en Drive — esto es lo único que persiste, y son solo números.
5. Tiene **checkpointing**: si se corta la sesión de Colab, podés volver a correr esta misma celda y va a saltear los clips ya procesados.

⚠️ Con 9068 clips esto puede tardar bastante. Por eso primero corremos un **subset de prueba** para validar que todo funciona antes de lanzar el batch completo.

In [5]:
!pip install -q mediapipe

from google.colab import drive
drive.mount('/content/drive')

import os, glob, cv2, math
import numpy as np
import pandas as pd
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from tqdm.notebook import tqdm

PROJECT_DIR = '/content/drive/MyDrive/SparkleV2'
RAW_DATASET_DIR = '/content/daisee_raw/DAiSEE/DataSet'
LABELS_DIR = f'{PROJECT_DIR}/labels'
FEATURES_DIR = f'{PROJECT_DIR}/features'

for split in ['Train', 'Validation', 'Test']:
    os.makedirs(f'{FEATURES_DIR}/{split}', exist_ok=True)

print('Listo.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 10.0 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Listo.


## 1. Descargar el modelo de FaceLandmarker

La Tasks API no trae el modelo embebido, hay que descargarlo una vez (es un `.task` de Google, ~4MB, link oficial).

In [6]:
MODEL_PATH = '/content/face_landmarker.task'
if not os.path.exists(MODEL_PATH):
    !wget -q -O {MODEL_PATH} https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task

print('Modelo listo:', os.path.exists(MODEL_PATH), '-', os.path.getsize(MODEL_PATH), 'bytes')

Modelo listo: True - 3758596 bytes


## 2. Si el dataset raw no está en esta sesión, volver a bajarlo

Cada sesión nueva de Colab arranca con el disco `/content/` vacío. Si ya corriste la Fase 0+1 antes pero es una sesión nueva, corré esto para volver a bajar el dataset.

In [ ]:
if not os.path.exists(RAW_DATASET_DIR):
    print('No encontré el dataset en este runtime. Volviendo a descargar desde Kaggle...')
    from google.colab import files
    if not os.path.exists('/root/.kaggle/kaggle.json'):
        print('Subí de nuevo tu kaggle.json:')
        uploaded = files.upload()
        os.makedirs('/root/.kaggle', exist_ok=True)
        !cp kaggle.json /root/.kaggle/kaggle.json
        !chmod 600 /root/.kaggle/kaggle.json
    os.makedirs('/content/daisee_raw', exist_ok=True)
    !kaggle datasets download -d olgaparfenova/daisee -p /content/daisee_raw --unzip
else:
    print('Dataset ya presente en este runtime, seguimos.')

No encontré el dataset en este runtime. Volviendo a descargar desde Kaggle...
Subí de nuevo tu kaggle.json:


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/olgaparfenova/daisee
License(s): unknown
100% 14.3G/14.3G [01:11<00:00, 215MB/s]



## 3. Cargar labels y armar índice de rutas de video

Limpiamos el nombre de columna `Frustration ` (tiene un espacio de más) y armamos un diccionario `ClipID -> ruta completa del .avi`.

In [7]:
def load_labels(split):
    df = pd.read_csv(f'{LABELS_DIR}/{split}Labels.csv')
    df.columns = [c.strip() for c in df.columns]
    return df

labels = {
    'Train': load_labels('Train'),
    'Validation': load_labels('Validation'),
    'Test': load_labels('Test'),
}

for split, df in labels.items():
    print(f'{split}: {len(df)} clips con label')

labels['Train'].head()

Train: 5358 clips con label
Validation: 1429 clips con label
Test: 1784 clips con label


,ClipID,Boredom,Engagement,Confusion,Frustration
0,1100011002.avi,0,2,0,0
1,1100011003.avi,0,2,0,0
2,1100011004.avi,0,3,0,0
3,1100011005.avi,0,3,0,0
4,1100011006.avi,0,3,0,0


In [8]:
video_index = {}  # ClipID (con .avi) -> ruta completa

for split in ['Train', 'Validation', 'Test']:
    split_dir = f'{RAW_DATASET_DIR}/{split}'
    paths = glob.glob(f'{split_dir}/**/*.avi', recursive=True)
    for p in paths:
        clip_id = os.path.basename(p)
        video_index[clip_id] = p

print(f'Videos indexados en disco: {len(video_index)}')

Videos indexados en disco: 0


In [9]:
for split, df in labels.items():
    ids_con_label = set(df['ClipID'])
    ids_en_disco = set(video_index.keys())
    faltan_video = ids_con_label - ids_en_disco
    print(f'{split}: {len(ids_con_label)} labels | {len(faltan_video)} labels sin video físico')
    if faltan_video:
        print('  Ejemplos sin video:', list(faltan_video)[:3])

Train: 5358 labels | 5358 labels sin video físico
  Ejemplos sin video: ['2100522013.avi', '4599990221.avi', '4599990137.avi']
Validation: 1429 labels | 1429 labels sin video físico
  Ejemplos sin video: ['4000231008.avi', '4100191030.avi', '5674960170.avi']
Test: 1784 labels | 1784 labels sin video físico
  Ejemplos sin video: ['5100351040.avi', '5100471047.avi', '826412010.avi']


## 4. Funciones de extracción de features

- **EAR (Eye Aspect Ratio):** relación de apertura del ojo, usando 6 landmarks por ojo. Baja cuando el ojo se cierra (parpadeo/somnolencia).
- **Head pose (yaw, pitch, roll):** los sacamos de la matriz de transformación facial que devuelve `FaceLandmarker` (alineación del rostro contra un modelo canónico vía Procrustes — más robusto que un `solvePnP` con puntos genéricos).

In [10]:
# Índices de landmarks (malla de 468 puntos, se mantienen igual entre la API legacy y la Tasks API)
RIGHT_EYE = [33, 160, 158, 133, 153, 144]
LEFT_EYE  = [362, 385, 387, 263, 373, 380]

FEATURE_NAMES = ['ear_avg', 'ear_left', 'ear_right', 'yaw', 'pitch']
N_FRAMES_PER_CLIP = 60


def create_face_landmarker():
    base_options = mp_python.BaseOptions(model_asset_path=MODEL_PATH)
    options = mp_vision.FaceLandmarkerOptions(
        base_options=base_options,
        running_mode=mp_vision.RunningMode.IMAGE,
        num_faces=1,
        min_face_detection_confidence=0.5,
        output_facial_transformation_matrixes=True,
    )
    return mp_vision.FaceLandmarker.create_from_options(options)


def eye_aspect_ratio(landmarks, eye_ids, w, h):
    pts = np.array([(landmarks[i].x * w, landmarks[i].y * h) for i in eye_ids])
    vert1 = np.linalg.norm(pts[1] - pts[5])
    vert2 = np.linalg.norm(pts[2] - pts[4])
    horiz = np.linalg.norm(pts[0] - pts[3])
    if horiz == 0:
        return np.nan
    return (vert1 + vert2) / (2.0 * horiz)


def euler_from_transformation_matrix(matrix):
    rot = np.array(matrix)[:3, :3]
    sy = math.sqrt(rot[0, 0] ** 2 + rot[1, 0] ** 2)
    singular = sy < 1e-6
    if not singular:
        pitch = math.degrees(math.atan2(rot[2, 1], rot[2, 2]))
        yaw = math.degrees(math.atan2(-rot[2, 0], sy))
        roll = math.degrees(math.atan2(rot[1, 0], rot[0, 0]))
    else:
        pitch = math.degrees(math.atan2(-rot[1, 2], rot[1, 1]))
        yaw = math.degrees(math.atan2(-rot[2, 0], sy))
        roll = 0.0
    return yaw, pitch, roll


def extract_features_from_frame(detector, frame_bgr):
    h, w = frame_bgr.shape[:2]
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
    result = detector.detect(mp_image)

    if not result.face_landmarks:
        return np.array([np.nan] * 5)

    landmarks = result.face_landmarks[0]
    ear_r = eye_aspect_ratio(landmarks, RIGHT_EYE, w, h)
    ear_l = eye_aspect_ratio(landmarks, LEFT_EYE, w, h)
    ear_avg = (ear_r + ear_l) / 2.0

    yaw, pitch = np.nan, np.nan
    if result.facial_transformation_matrixes:
        yaw, pitch, _roll = euler_from_transformation_matrix(
            result.facial_transformation_matrixes[0]
        )

    return np.array([ear_avg, ear_l, ear_r, yaw, pitch], dtype=np.float64)

## 5. Función de procesamiento por clip

In [11]:
def process_clip(video_path, detector, n_frames=N_FRAMES_PER_CLIP):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames <= 0:
        cap.release()
        return None

    frame_indices = np.linspace(0, total_frames - 1, n_frames).astype(int)
    sequence = np.full((n_frames, len(FEATURE_NAMES)), np.nan)

    for seq_pos, frame_idx in enumerate(frame_indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ret, frame = cap.read()
        if not ret:
            continue
        sequence[seq_pos] = extract_features_from_frame(detector, frame)

    cap.release()
    return sequence

## 6. Prueba en un subset chico (validar antes de correr todo)

Corremos sobre 20 clips de Train para chequear que todo funciona y ver cuánto tarda por clip.

In [12]:
import time

test_clips = labels['Train'].head(20)
detector = create_face_landmarker()

try:
    t0 = time.time()
    resultados = []
    for _, row in test_clips.iterrows():
        clip_id = row['ClipID']
        if clip_id not in video_index:
            print(f'{clip_id}: sin video físico, salteado')
            continue
        seq = process_clip(video_index[clip_id], detector)
        pct_nan = np.isnan(seq).any(axis=1).mean() * 100 if seq is not None else 100
        resultados.append((clip_id, pct_nan))
        print(f'{clip_id}: {pct_nan:.1f}% frames sin rostro detectado')
    elapsed = time.time() - t0
finally:
    detector.close()

print(f'\nTiempo total: {elapsed:.1f}s para {len(resultados)} clips ({elapsed/max(len(resultados),1):.2f}s/clip)')
print(f'Estimado para el dataset completo (~9068 clips): {elapsed/max(len(resultados),1)*9068/3600:.1f} horas')

1100011002.avi: sin video físico, salteado
1100011003.avi: sin video físico, salteado
1100011004.avi: sin video físico, salteado
1100011005.avi: sin video físico, salteado
1100011006.avi: sin video físico, salteado
1100011007.avi: sin video físico, salteado
1100011008.avi: sin video físico, salteado
1100011009.avi: sin video físico, salteado
1100011010.avi: sin video físico, salteado
1100011011.avi: sin video físico, salteado
1100011012.avi: sin video físico, salteado
1100011013.avi: sin video físico, salteado
1100011014.avi: sin video físico, salteado
1100011015.avi: sin video físico, salteado
1100011016.avi: sin video físico, salteado
1100011017.avi: sin video físico, salteado
1100011018.avi: sin video físico, salteado
1100011019.avi: sin video físico, salteado
1100011020.avi: sin video físico, salteado
1100011021.avi: sin video físico, salteado

Tiempo total: 0.0s para 0 clips (0.00s/clip)
Estimado para el dataset completo (~9068 clips): 0.0 horas


**Antes de seguir:** revisá el `% frames sin rostro detectado` de cada clip. Si ves porcentajes muy altos (>50%) en la mayoría, avisame antes de lanzar el batch completo.

Con la estimación de tiempo vamos a decidir si conviene: (a) correr todo en una sesión larga, (b) recortar `N_FRAMES_PER_CLIP`, o (c) correr por tandas en varias sesiones (el checkpointing de la sección 7 soporta esto).

## 7. Batch completo con checkpointing

Procesa todos los clips de un split. Si un `.npz` ya existe en Drive para un `ClipID`, lo saltea — así podés cortar la sesión de Colab y retomar después sin perder lo ya hecho.

Corré esta celda una vez por split (`Train`, `Validation`, `Test`), cambiando la variable `SPLIT` de abajo.

In [13]:
SPLIT = 'Test'  # cambiar a 'Validation' o 'Test' cuando corresponda

df_split = labels[SPLIT]
out_dir = f'{FEATURES_DIR}/{SPLIT}'
log_path = f'{FEATURES_DIR}/{SPLIT}_processing_log.csv'

log_rows = []
if os.path.exists(log_path):
    log_rows = pd.read_csv(log_path).to_dict('records')

already_done = set(os.path.splitext(f)[0] for f in os.listdir(out_dir) if f.endswith('.npz'))
print(f'{SPLIT}: {len(already_done)} clips ya procesados de {len(df_split)} totales')

Test: 1638 clips ya procesados de 1784 totales


In [17]:
detector = create_face_landmarker()

try:
    for _, row in tqdm(df_split.iterrows(), total=len(df_split)):
        clip_id = row['ClipID']
        clip_key = os.path.splitext(clip_id)[0]

        if clip_key in already_done:
            continue
        if clip_id not in video_index:
            log_rows.append({'ClipID': clip_id, 'status': 'sin_video_fisico', 'pct_nan': None})
            continue

        try:
            seq = process_clip(video_index[clip_id], detector)
            if seq is None:
                log_rows.append({'ClipID': clip_id, 'status': 'video_ilegible', 'pct_nan': None})
                continue

            pct_nan = np.isnan(seq).any(axis=1).mean() * 100

            np.savez_compressed(
                f'{out_dir}/{clip_key}.npz',
                sequence=seq,
                feature_names=FEATURE_NAMES,
                boredom=row['Boredom'],
                engagement=row['Engagement'],
                confusion=row['Confusion'],
                frustration=row['Frustration'],
            )
            log_rows.append({'ClipID': clip_id, 'status': 'ok', 'pct_nan': pct_nan})

        except Exception as e:
            log_rows.append({'ClipID': clip_id, 'status': f'error: {e}', 'pct_nan': None})

        if len(log_rows) % 50 == 0:
            pd.DataFrame(log_rows).to_csv(log_path, index=False)
finally:
    detector.close()

pd.DataFrame(log_rows).to_csv(log_path, index=False)
print('Terminado (o cortado). Progreso guardado en', log_path)

  0%|          | 0/1784 [00:00<?, ?it/s]

Terminado (o cortado). Progreso guardado en /content/drive/MyDrive/SparkleV2/features/Test_processing_log.csv


## 8. Resumen del procesamiento

---



In [14]:
log_df = pd.read_csv(log_path)
print(log_df['status'].value_counts())

ok_df = log_df[log_df['status'] == 'ok']
if len(ok_df):
    print(f"\n% NaN promedio entre clips OK: {ok_df['pct_nan'].mean():.2f}%")
    print(f"Clips con >30% frames sin rostro: {(ok_df['pct_nan'] > 30).sum()}")

status
ok                  1638
sin_video_fisico     146
Name: count, dtype: int64

% NaN promedio entre clips OK: 0.07%
Clips con >30% frames sin rostro: 0


In [2]:
# 1. Volver a montar Google Drive (obligatorio porque se desconectó el entorno)
from google.colab import drive
drive.mount('/content/drive')

# 2. Importar pandas de nuevo
import pandas as pd

# 3. Definir rutas y leer el log que quedó guardado en tu Drive
PROJECT_DIR = '/content/drive/MyDrive/SparkleV2'
test_log_path = f'{PROJECT_DIR}/features/Test_processing_log.csv'

try:
    log_df = pd.read_csv(test_log_path)
    print("=========================================")
    print("      RESUMEN DEL SPLIT DE TEST")
    print("=========================================\n")
    print(log_df['status'].value_counts())

    ok_df = log_df[log_df['status'] == 'ok']
    if len(ok_df):
        print(f"\n% NaN promedio entre clips OK: {ok_df['pct_nan'].mean():.2f}%")
        print(f"Clips con >30% frames sin rostro: {(ok_df['pct_nan'] > 30).sum()}")
except FileNotFoundError:
    print(f"⚠️ No se encontró el archivo en: {test_log_path}")
    print("Verificá en tu Google Drive si el archivo 'Test_processing_log.csv' existe dentro de SparkleV2/features.")

Mounted at /content/drive
      RESUMEN DEL SPLIT DE TEST

status
ok                  1638
sin_video_fisico     146
Name: count, dtype: int64

% NaN promedio entre clips OK: 0.07%
Clips con >30% frames sin rostro: 0


## Checkpoint de la Fase 2+3

Repetí la sección 7 cambiando `SPLIT` para `Validation` y `Test`.

Cuando tengas el subset de prueba corrido (sección 6), pegame acá:
- El tiempo estimado total que dio esa celda.
- El `% frames sin rostro detectado` de esos 20 clips.

Con eso decidimos si conviene ajustar `N_FRAMES_PER_CLIP` antes de lanzar el batch completo, y armamos la Fase 4 (tratamiento de faltantes) + Fase 5 (armado del dataset secuencial final).